# 1.4 Demand Pipeline: H0 -> G0 -> Elasticity Recovery -> A/B Simulation

This notebook is organized around the four-step research design:

```text
Step 1  H0:   validate and calibrate (anchor + measured multipliers)
Step 2  G0:   Maria aggregate/share model -> Glamsterdam equilibrium reference
Step 3  Recover own-price elasticities and reference quantities from G0
Step 4  A/B:  simulate separate-resource worlds with those own-price
              elasticities + swept data elasticity + measured bundle coupling
```

Offline (cached CSVs + `src/demand/`, 110 tests). Every calibration ingredient
prints its provenance. BAL bytes come from the notebook-1.3 regression
predictions (`calibration_bal_block_predictions_*.csv`).

In [ ]:
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from demand import (
    SHARE_MODES,
    Anchor,
    CrossElasticityDemand,
    DemandParams,
    FeeDimension,
    WorldSpec,
    anchor_from_equilibrium,
    combine_matrices,
    coupling_exposures_from_transactions,
    coupling_matrix,
    implied_anchor_elasticities,
    implied_own_price_elasticities,
    share_model_jacobian,
    solve_equilibrium,
)

pd.options.display.float_format = "{:,.4f}".format
plt.style.use("seaborn-v0_8-whitegrid")

START_DATE = "2026-02-01"
END_DATE = "2026-06-01"
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

PANEL_CSV = DATA_DIR / f"daily_accounting_panel_{START_DATE}_{END_DATE}.csv"
BAL_PRED_CSV = DATA_DIR / f"calibration_bal_block_predictions_{START_DATE}_{END_DATE}.csv"
BAL_BLOCKS_CSV = DATA_DIR / f"calibration_rpc_bal_blocks_{START_DATE}_{END_DATE}.csv"
STATE_ACCESS_AUTH_CSV = DATA_DIR / f"calibration_rpc_state_access_auth_blocks_{START_DATE}_{END_DATE}.csv"
RPC_BLOCKS_CSV = DATA_DIR / f"calibration_rpc_blocks_{START_DATE}_{END_DATE}.csv"
PILOT_TX_CSV = DATA_DIR / "glamsterdam_tx_regular_gas_repriced_24120001_24120500.csv"
PILOT_BAL_CSV = DATA_DIR / "pilot_per_tx_bal_24120001_24120500.csv"
AUTH_RECORDS_CSV = DATA_DIR / "rpc_authorization_records_24120001_24120500.csv"
OUT_RESULTS_CSV = DATA_DIR / f"demand_equilibrium_results_{START_DATE}_{END_DATE}.csv"

panel = pd.read_csv(PANEL_CSV)
print("panel days:", len(panel), "| blocks:", int(panel["block_count"].sum()))

## Step 1 — H0: Validate And Calibrate

The H0 anchor is today's observed demand: level (`T_ref`), price (`p_ref`), and
composition, in **old-gas physical units** (state = current state-creation gas,
data = standard calldata gas, execution = the rest). H0 accounting was
validated receipt-anchored on the pilot window (per-tx gas sums exactly to
block gas).

Calibration ingredients use sampled-block means; BAL uses the 1.3 regression
predictions across all sampled blocks.

**Where BALs enter — and where they don't.** BAL/AL/auth bytes have *no* cost
at H0 (BALs are unpriced until the 7999 data resource exists), so they are
correctly absent from the H0 anchor shares and from the decomposition weights
of Maria's rest elasticity (estimated on data where they were free). They
enter the A/B worlds through the data metering multiplier
`m_data_7999 = 16 x (content bytes incl BAL) / old calldata gas`, which is
what lifts data's weight within "rest" from ~4% (H0 decomposition weight) to
~15% (B-world metering weight) — both printed below.

In [ ]:
blocks_total = panel["block_count"].sum()
T_REF = float(panel["current_gas_used"].sum() / blocks_total)
P_REF = float(panel["median_base_fee_per_gas"].median())

share_state = float(panel["current_state_creation_gas"].sum() / panel["current_gas_used"].sum())
share_data = float(panel["standard_calldata_gas"].sum() / panel["current_gas_used"].sum())
share_exec = 1.0 - share_state - share_data

anchor_h0 = Anchor.from_single_price(
    total_gas=T_REF,
    base_fee_wei=P_REF,
    shares={"execution": share_exec, "data": share_data, "state": share_state},
)
print(f"T_ref = {T_REF/1e6:.2f}M gas/block | p_ref = {P_REF/1e9:.4f} gwei")
print(f"H0 shares: execution {share_exec:.3f} | data {share_data:.3f} | state {share_state:.3f}")


def sampled_mean(sources, fallback, label):
    for path, column in sources:
        if path.exists():
            frame = pd.read_csv(path)
            if column in frame.columns and frame[column].notna().any():
                series = frame[column].dropna()
                se = float(series.std() / np.sqrt(len(series)))
                return float(series.mean()), f"sampled ({len(series):,} blocks, {column})", se
    return float(fallback), f"PILOT FALLBACK ({label})", float("nan")


AL_GAS_7981, al_gas_src, al_se = sampled_mean(
    [(STATE_ACCESS_AUTH_CSV, "tx_access_list_gas_7981"), (RPC_BLOCKS_CSV, "tx_access_list_gas_7981")],
    395_275, "0.40M gas/block")
AL_BYTES, al_bytes_src, al_b_se = sampled_mean(
    [(STATE_ACCESS_AUTH_CSV, "tx_access_list_bytes"), (RPC_BLOCKS_CSV, "tx_access_list_bytes")],
    6_176, "6.2KB/block")
AUTH_BYTES, auth_src, auth_se = sampled_mean(
    [(STATE_ACCESS_AUTH_CSV, "authorization_tuple_8131_bytes"), (RPC_BLOCKS_CSV, "authorization_tuple_8131_bytes")],
    138, "138B/block")
BAL_BYTES, bal_src, bal_se = sampled_mean(
    [(BAL_PRED_CSV, "bal_rlp_bytes_pred"), (BAL_BLOCKS_CSV, "bal_rlp_bytes"), (RPC_BLOCKS_CSV, "bal_rlp_bytes")],
    148_168, "148KB/block")

known_bytes_per_block = float(panel["bandwidth_known_bytes_no_bal"].sum() / blocks_total)
old_calldata_gas_per_block = float(panel["standard_calldata_gas"].sum() / blocks_total)
floor_uplift_per_block = float(panel["eip7976_floor_uplift_current_body"].sum() / blocks_total)

M_STATE = float(panel["state_gas_8037"].sum() / panel["current_state_creation_gas"].sum())
M_DATA_G0 = (old_calldata_gas_per_block + floor_uplift_per_block + AL_GAS_7981) / old_calldata_gas_per_block
data_bytes_total = known_bytes_per_block + AL_BYTES + AUTH_BYTES + BAL_BYTES
M_DATA_7999 = 16.0 * data_bytes_total / old_calldata_gas_per_block

provenance = pd.DataFrame([
    {"ingredient": "m_state (8037 repricing)", "value": M_STATE, "se": float("nan"), "source": "panel (full window)"},
    {"ingredient": "EIP-7981 AL gas/block", "value": AL_GAS_7981, "se": al_se, "source": al_gas_src},
    {"ingredient": "AL bytes/block", "value": AL_BYTES, "se": al_b_se, "source": al_bytes_src},
    {"ingredient": "auth 8131 bytes/block", "value": AUTH_BYTES, "se": auth_se, "source": auth_src},
    {"ingredient": "BAL RLP bytes/block", "value": BAL_BYTES, "se": bal_se, "source": bal_src},
    {"ingredient": "m_data_G0 (floors + 7981, no BAL)", "value": M_DATA_G0, "se": float("nan"), "source": "derived"},
    {"ingredient": "m_data_7999 (16/byte incl BAL)", "value": M_DATA_7999, "se": float("nan"), "source": "derived"},
])
display(provenance)

w_d_rest_h0 = anchor_h0.data_share_of_rest
w_d_rest_b = (w_d_rest_h0 * M_DATA_7999) / ((1 - w_d_rest_h0) + w_d_rest_h0 * M_DATA_7999)
print(f"data weight within rest: H0 decomposition weight {w_d_rest_h0:.3f} "
      f"| B-world metering weight {w_d_rest_b:.3f} (BALs/AL/auth via m_data_7999)")

## Step 2 — G0: The Glamsterdam Equilibrium Reference

Maria's aggregate + share system, solved under the Glamsterdam mechanism
(8037 state metering, 7976 floor + 7981 surcharge in regular gas, BALs
present but unpriced, `max(regular, state)` at 100M/50M). Solved under both
share modes because her one-price eta cannot distinguish them — under
Glamsterdam, state's *absolute* effective price falls (~0.5x, fee collapse
beats the 5.7x repricing) while its *relative* price rises (~5.4x), so the
modes disagree in sign on state growth. Mode disagreement is a reported
finding.

In [ ]:
G0_TARGET = 50_000_000
MULT_G0 = {"execution": 1.0, "data": M_DATA_G0, "state": M_STATE}
MULT_7999 = {"execution": 1.0, "data": M_DATA_7999, "state": M_STATE}

world_g0 = WorldSpec(
    name="G0_glamsterdam",
    multipliers=MULT_G0,
    dimensions=(
        FeeDimension(name="regular_state", groups=(("execution", "data"), ("state",)), target_gas=G0_TARGET),
    ),
)

MODES = list(SHARE_MODES)
PARAMS = {mode: DemandParams(eps_agg=0.175, eta_state=0.43, share_mode=mode) for mode in MODES}

eq_g0 = {}
anchor_g0 = {}
for mode in MODES:
    eq = solve_equilibrium(anchor_h0, PARAMS[mode], world_g0)
    eq_g0[mode] = eq
    anchor_g0[mode] = anchor_from_equilibrium(anchor_h0, world_g0, eq)
    fee = eq.prices_wei["regular_state"]
    print(f"[{mode}] fee x{fee/P_REF:.3f} | T x{eq.demand_expansion:.2f} | "
          f"state share {anchor_h0.alpha_state:.3f} -> {eq.alpha_state:.3f} | "
          f"physical state x{eq.quantities['state']/anchor_h0.quantities['state']:.2f} | "
          f"binding: {'state' if eq.binding_group['regular_state'] == 1 else 'regular'}")

## Step 3 — Recover Own-Price Elasticities At G0

The recovery is `implied_anchor_elasticities` (and the full matrix,
`share_model_jacobian`) evaluated at the **G0 anchor**. Read the caveat
correctly: eps_agg = 0.175 and eta = 0.43 are **Maria's H0 estimates
transported unchanged** — the 7981/7976/8037/BAL repricings enter through the
accounting and the mix shift (alpha), not through re-estimation (G0 is a
counterfactual with no observed price response). So the state number is
"Maria's eta at the Glamsterdam mix", not a Glamsterdam-estimated elasticity.

Splitting rest into execution vs data uses the **H0 decomposition weights**
(~96/4 — BALs correctly absent: they were unpriced when the elasticity was
measured). Because data is so small a slice, the split pins execution at
~0.08 while leaving the data elasticity nearly unconstrained — data needs its
own identification (EIP-7623 event, blob spillover).

In [ ]:
rows = []
for mode in MODES:
    implied = implied_anchor_elasticities(anchor_g0[mode], PARAMS[mode])
    rows.append({"share_mode": mode, "anchor": "G0",
                 "alpha_state": anchor_g0[mode].alpha_state,
                 "eps_state": implied["state"], "eps_rest": implied["rest"]})
    implied_h0 = implied_anchor_elasticities(anchor_h0, PARAMS[mode])
    rows.append({"share_mode": mode, "anchor": "H0",
                 "alpha_state": anchor_h0.alpha_state,
                 "eps_state": implied_h0["state"], "eps_rest": implied_h0["rest"]})
recovery = pd.DataFrame(rows)
display(recovery)

eps_rest_g0 = float(recovery[(recovery.share_mode == "maria_own_price") & (recovery.anchor == "G0")]["eps_rest"].iloc[0])
w_exec = 1 - w_d_rest_h0
split_rows = []
for eps_data in [0.02, 0.08, 0.15, 0.30]:
    split_rows.append({"assumed_eps_data": eps_data,
                       "implied_eps_execution": (eps_rest_g0 - w_d_rest_h0 * eps_data) / w_exec})
print(f"rest split (H0 weights: execution {w_exec:.3f} / data {w_d_rest_h0:.3f}; eps_rest = {eps_rest_g0:.3f}):")
display(pd.DataFrame(split_rows))
print("=> execution pinned ~0.08 regardless; eps_data must come from the 7623 event / blob spillover (swept until then)")

## Step 4 — A/B Simulation: Own-Price Elasticities + Bundle Coupling

The share model's cross-price effects are pure **substitution** (positive).
Transactions also create **complementarity** (negative cross effects): pricing
out a state-heavy transaction removes the execution and data it carried. Step
4 recomposes demand as a constant-elasticity matrix around the G0 anchor:

```text
Q_i = Q_i_ref * prod_j (r_j ^ E[i][j])

E = share_model_jacobian(G0 anchor)          # substitution (Maria, transported)
  + coupling_matrix(exposures, own_price)     # complementarity (measured)

gamma[i][j] = -eps_own[j] * phi[i][j]
phi[i][j]   = share of resource i riding in j-intensive transactions
```

The exposures phi are **measured** from the pilot per-transaction
decomposition, with every resource in **world-B metered gas** so the within-tx
cost shares that weight the exposures are B's actual fee burden: execution =
old-gas branch, state = 8037 bytes x CPSB, and **data = 16 x all priced
content bytes (calldata + access list + auth tuples + BAL)**. Including the
non-calldata content matters most for BAL: BAL bytes are a byproduct of state
*access*, unpriced today, that the 7999 data resource meters — so they add a
real execution/state <-> data coupling that a calldata-only column misses. The
cell prints the calldata-only exposures alongside for that comparison.

BAL is attributed to transactions from the block BAL (each write/balance/
nonce/code change carries its tx index), which is available on a sampled
subset of pilot blocks, so the exposures are measured on that subset (size
printed). With coupling off, the matrix reproduces the share model near the
anchor (tested).

In [ ]:
CPSB = 1530
tx = pd.read_csv(PILOT_TX_CSV)

# Per-tx resource quantities in world-B metered gas (see markdown).
tx["execution_gas"] = (tx["standard_branch_after_state"] - tx["standard_calldata_gas"]).clip(lower=0)
tx["state_gas"] = (tx["new_storage_slots"] * 64 + tx["new_accounts"] * 120 + tx["code_bytes"]) * CPSB

auth_by_tx = (
    pd.read_csv(AUTH_RECORDS_CSV)
    .groupby(["block_number", "tx_index"], as_index=False)["authorization_tuple_8131_bytes"].sum()
)
tx = tx.merge(auth_by_tx, on=["block_number", "tx_index"], how="left")
tx["authorization_tuple_8131_bytes"] = tx["authorization_tuple_8131_bytes"].fillna(0.0)

if PILOT_BAL_CSV.exists():
    tx = tx.merge(pd.read_csv(PILOT_BAL_CSV), on=["block_number", "tx_index"], how="inner")
    scope = f"{tx['block_number'].nunique()} pilot blocks with per-tx BAL, {len(tx):,} txs"
else:
    tx["bal_rlp_bytes"] = 0.0
    scope = f"pilot window, {len(tx):,} txs (per-tx BAL cache missing -> BAL excluded)"

data_bytes = (tx["calldata_bytes"] + tx["access_list_bytes"]
              + tx["authorization_tuple_8131_bytes"] + tx["bal_rlp_bytes"])
tx["data_gas"] = 16.0 * data_bytes
tx["data_gas_calldata_only"] = 16.0 * tx["calldata_bytes"]

exposures = coupling_exposures_from_transactions(
    tx, execution_col="execution_gas", data_col="data_gas", state_col="state_gas")
calldata_only = coupling_exposures_from_transactions(
    tx, execution_col="execution_gas", data_col="data_gas_calldata_only", state_col="state_gas")

exposure_frame = pd.DataFrame(exposures).T
exposure_frame.index.name = "resource i (rides)"
exposure_frame.columns.name = "j-intensity"
print(f"measured bundle exposures phi[i][j] ({scope}):")
display(exposure_frame)
denom = float(data_bytes.sum())
print(f"data content bytes: calldata {tx['calldata_bytes'].sum()/denom:.2f} | "
      f"access list {tx['access_list_bytes'].sum()/denom:.2f} | "
      f"auth {tx['authorization_tuple_8131_bytes'].sum()/denom:.2f} | "
      f"BAL {tx['bal_rlp_bytes'].sum()/denom:.2f}")
print(f"BAL/AL/auth lift phi[execution][data] {calldata_only['execution']['data']:.3f} -> "
      f"{exposures['execution']['data']:.3f}, phi[state][data] "
      f"{calldata_only['state']['data']:.3f} -> {exposures['state']['data']:.3f}")

B_EXEC_TARGET = 225_000_000
DATA_TARGET = 15_000_000
STATE_TARGET = 75_000_000
world_b = WorldSpec(
    name="B_full_7999",
    multipliers=MULT_7999,
    dimensions=(
        FeeDimension(name="execution", groups=(("execution",),), target_gas=B_EXEC_TARGET),
        FeeDimension(name="data", groups=(("data",),), target_gas=DATA_TARGET),
        FeeDimension(name="state", groups=(("state",),), target_gas=STATE_TARGET),
    ),
)
world_a = WorldSpec(
    name="A_8037_plus_data",
    multipliers=MULT_7999,
    dimensions=(
        FeeDimension(name="execution_state", groups=(("execution",), ("state",)), target_gas=B_EXEC_TARGET),
        FeeDimension(name="data", groups=(("data",),), target_gas=DATA_TARGET),
    ),
)

b_rows = []
for mode in MODES:
    jac = share_model_jacobian(anchor_g0[mode], PARAMS[mode])
    own = implied_own_price_elasticities(jac)
    gamma = coupling_matrix(exposures, own)
    coupled = CrossElasticityDemand(anchor=anchor_g0[mode], matrix=combine_matrices(jac, gamma))
    uncoupled = CrossElasticityDemand(anchor=anchor_g0[mode], matrix=jac)

    for label, demand_fn in [
        ("share_model", None),
        ("matrix_no_coupling", uncoupled.quantities),
        ("matrix_with_coupling", coupled.quantities),
    ]:
        eq = solve_equilibrium(anchor_g0[mode], PARAMS[mode], world_b, demand_fn=demand_fn)
        row = {"share_mode": mode, "demand": label, "converged": eq.converged}
        for dim in ("execution", "data", "state"):
            row[f"{dim}_usage_M"] = eq.metered_usage[dim] / 1e6
            row[f"{dim}_floor"] = eq.floor_binding[dim]
            row[f"{dim}_fee_gwei"] = eq.prices_wei[dim] / 1e9
        row["state_physical_x_today"] = eq.quantities["state"] / anchor_h0.quantities["state"]
        b_rows.append(row)

b_results = pd.DataFrame(b_rows)
display(b_results)
b_results.to_csv(OUT_RESULTS_CSV, index=False)
print("wrote", OUT_RESULTS_CSV)

### Sweeps: eta_state, eta_data (world B, matrix with coupling)

`eta_state` sweeps how much of the recovered 0.51 is real own-price response
vs borrowed substitution. `eta_data` sweeps the structural data elasticity:
the daily estimate is ~0.02 (near-complement, matches Offchain's L2
differential); 0.1-0.3 are what-ifs pending the EIP-7623 event study.

In [ ]:
sweep_rows = []
for mode in MODES:
    for eta_state in [0.0, 0.2, 0.43, 0.8]:
        for eta_data in [0.02, 0.1, 0.3]:
            params = DemandParams(eps_agg=0.175, eta_state=eta_state, eta_data=eta_data, share_mode=mode)
            jac = share_model_jacobian(anchor_g0[mode], params)
            own = implied_own_price_elasticities(jac)
            model = CrossElasticityDemand(
                anchor=anchor_g0[mode],
                matrix=combine_matrices(jac, coupling_matrix(exposures, own)),
            )
            eq = solve_equilibrium(anchor_g0[mode], params, world_b, demand_fn=model.quantities)
            sweep_rows.append({
                "share_mode": mode, "eta_state": eta_state, "eta_data": eta_data,
                "state_usage_M": eq.metered_usage["state"] / 1e6,
                "state_floor": eq.floor_binding["state"],
                "data_usage_M": eq.metered_usage["data"] / 1e6,
                "data_floor": eq.floor_binding["data"],
                "exec_floor": eq.floor_binding["execution"],
                "state_physical_x": eq.quantities["state"] / anchor_h0.quantities["state"],
            })
sweeps = pd.DataFrame(sweep_rows)
display(sweeps[sweeps["eta_data"] == 0.02])

fig, ax = plt.subplots(figsize=(8.5, 4.8))
styles = {"maria_own_price": "-", "relative_state_vs_rest": "--"}
for mode in MODES:
    for eta_data in [0.02, 0.3]:
        sub = sweeps[(sweeps["share_mode"] == mode) & (sweeps["eta_data"] == eta_data)]
        ax.plot(sub["eta_state"], sub["state_physical_x"], styles[mode], marker="o",
                label=f"{mode.split('_')[0]} eta_d={eta_data}")
ax.axhline(1.0, color="black", linewidth=1, linestyle=":")
ax.set_xlabel("eta_state")
ax.set_ylabel("physical state creation, x today")
ax.set_title("World B state growth (matrix + measured coupling)")
ax.legend(fontsize=8, ncols=2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / f"demand_b_state_growth_sweep_{START_DATE}_{END_DATE}.png", dpi=160)
plt.show()
sweeps.to_csv(DATA_DIR / f"demand_b_sweep_grid_{START_DATE}_{END_DATE}.csv", index=False)

### Capacity check: the governor rule at 450M-limit ambitions

In [ ]:
for mode in MODES:
    alpha = eq_g0[mode].alpha_state
    w_d = anchor_g0[mode].data_share_of_rest
    intensity = {
        "execution": (1 - alpha) * (1 - w_d) * 1.0,
        "data": (1 - alpha) * w_d * M_DATA_7999,
        "state": alpha * M_STATE,
    }
    caps = {name: tgt / intensity[name] / 1e6
            for name, tgt in [("execution", 225e6), ("data", DATA_TARGET), ("state", STATE_TARGET)]}
    governor = min(caps, key=caps.get)
    print(f"[{mode}] T-caps (M): " + ", ".join(f"{k} {v:.0f}" for k, v in caps.items())
          + f" -> governor: {governor}")

## Caveats

- **Step 3's numbers are transported, not re-estimated.** eps_agg and eta are
  Maria's H0 estimates; Glamsterdam only shifts the mix they are evaluated at.
  G0 has no observed price response to estimate from.
- **The state 0.51 carries the share model's substitution assumption**; the
  eta_state sweep is the honesty band. The execution ~0.08 is well-pinned
  (execution dominates rest); the data elasticity is nearly unconstrained by
  Maria's numbers and needs the EIP-7623 event / blob-spillover estimates.
- **Coupling v1 assumptions**: exposures measured at the pilot window's H0 mix
  (composition of the *marginal* transaction may differ — the tx-clustering
  stretch goal refines this); the matrix freezes elasticities at the G0
  anchor (share model lets them drift, agreement ~3% after ~2x price moves,
  tested); exposures + substitution can double-move if eta is also large —
  the sweep covers it.
- **Share-mode disagreement remains the headline uncertainty** (own-price vs
  relative reading of Maria's eta; opposite signs on G0 state growth). The
  7623 event and the first post-8037 months are the adjudicating variation.
- **BAL/AL/auth**: absent from H0 shares and rest-decomposition weights
  (correctly — unpriced at estimation time); present in A/B via
  m_data_7999 (block level) and in the Step-4 coupling data column at the
  transaction level — calldata + access list + auth tuples + per-tx-attributed
  BAL, all metered at 16 gas/byte. Per-tx BAL is attributed from the block BAL
  on a sampled subset of pilot blocks; the calldata-only exposures are printed
  alongside so the added coupling is auditable. BAL is a byproduct of state
  access (not independently demanded), so its main effect is a real
  execution/state <-> data complementarity.
- **Execution target**: A/B size the execution dimension at 225M gas
  (450M limit); state (75M) and data (15M) keep their own targets in B, while
  A bundles execution and state under the single 225M 8037 limit.